# Part 7 — A second venue: Bybit (Paper 2)
Every result so far is from Binance. This notebook tests the thin-book result on **Bybit**, a different exchange with different participants, using Bybit's free public trade archive. Bybit has never been used in this project.

Bybit's free archive has trades but **no order-book depth**, so the test uses the depth-free statistic C3_flow = κ²ηq − (1−a)κ. Parts 5–6 show that the depth term carries no forecasting content, and Part 6 also tests C3_flow on Binance.

**Rules:** run cells one at a time. **W3 downloads the pre-registration: e-mail it to yourself before W4.** Run W6 exactly once.

Expected time: about 1–2 hours (about 2,600 daily trade files).

In [ ]:
# ============================================================================
# CELL W0 — SETUP (self-contained; run first)
# Part 7: a SECOND VENUE. Bybit USDT perpetuals from Bybit's free trade archive (public.bybit.com),
# never used before. Bybit's free archive has trades but no depth history, so the test uses the
# depth-free amplification statistic C3_flow = kappa^2 eta q - (1-a) kappa (Parts 5-6 show the depth
# term carries no forecasting content). Same symbols as the Binance thin set, a different venue.
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, zlib
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

IN_COLAB = "google.colab" in sys.modules
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.filterwarnings("ignore", category=RuntimeWarning)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
XDATA = os.path.join(BASE, "data_xsec")
for _d in (RESULTS, PREREG, XDATA):
    os.makedirs(_d, exist_ok=True)

SEED = 20261022
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

def boot_ci(v, R, B=2000, stat=np.mean):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 3:
        return [np.nan, np.nan]
    bs = [stat(v[R.integers(0, len(v), len(v))]) for _ in range(B)]
    return [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]

# ---- Sample design --------------------------------------------------------------------------
BYBIT = "https://public.bybit.com/trading"
TRAIN_DAYS, TEST_DAYS = (10, 24), (3, 10, 17, 24)        # Bybit data were never used: all test days are fresh
TRAIN_START, TRAIN_END, TEST_START, TEST_END = "2023-01-01", "2024-12-31", "2025-01-01", "2026-09-30"
THREADS = 12
print("BASE =", BASE)
def _dates(a, b, days):
    return [d.strftime("%Y-%m-%d") for d in pd.date_range(a, b, freq="D") if d.day in days]
TRAIN_D, TEST_D = _dates(TRAIN_START, TRAIN_END, TRAIN_DAYS), _dates(TEST_START, TEST_END, TEST_DAYS)

def _http(url, method="GET", timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method=method, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, (r.read() if method == "GET" else b"")
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")
def bb_url(sym, date):
    return f"{BYBIT}/{sym}/{sym}{date}.csv.gz"
code, _ = _http(bb_url("BTCUSDT", "2024-06-10"), "HEAD", timeout=30)
print("Bybit archive reachable:", code == 200, f"(HTTP {code})")
assert code == 200, "public.bybit.com is not reachable from this runtime (Bybit blocks some regions); tell Claude."
DL_LOG = []


In [ ]:
# ============================================================================
# CELL W1 — FUNCTIONS (estimation code identical to Parts 4-6; only the data source differs)
# ============================================================================
import gzip
def build_day_bybit(sym, date):
    """Minute bars from Bybit trades: last price, signed taker notional (flow), notional volume ($M)."""
    code, raw = _http(bb_url(sym, date), timeout=180)
    if code != 200 or not raw:
        return None, {"symbol": sym, "date": date, "reason": f"http {code}"}
    DL_LOG.append({"symbol": sym, "date": date, "sha256": hashlib.sha256(raw).hexdigest(), "bytes": len(raw)})
    tr = pd.read_csv(io.BytesIO(gzip.decompress(raw)))
    tr.columns = [c.strip().lower() for c in tr.columns]
    ts = pd.to_numeric(tr["timestamp"], errors="coerce").to_numpy()
    ts = np.where(ts > 1e11, ts / 1000.0, ts)                   # seconds
    t0 = pd.Timestamp(date, tz="UTC").timestamp()
    m = ((ts - t0) // 60).astype("int64")
    px = pd.to_numeric(tr["price"], errors="coerce").to_numpy(); sz = pd.to_numeric(tr["size"], errors="coerce").to_numpy()
    notional = px * sz / 1e6; sign = np.where(tr["side"].astype(str).str.lower().str.startswith("b"), 1.0, -1.0)
    ok = (m >= 0) & (m < 1440) & np.isfinite(px) & np.isfinite(notional)
    T = pd.DataFrame({"m": m[ok], "ts": ts[ok], "px": px[ok], "flow": (sign * notional)[ok], "vol": notional[ok]}).sort_values("ts")
    idx = np.arange(1440)
    g = T.groupby("m")
    out = pd.DataFrame(index=idx)
    out["close"] = g["px"].last().reindex(idx).ffill().bfill()
    out["flow"] = g["flow"].sum().reindex(idx).fillna(0.0)
    out["vol"] = g["vol"].sum().reindex(idx).fillna(0.0)
    cover = float(g.size().reindex(idx).notna().mean())
    st = {"symbol": sym, "date": date, "n_trades": int(ok.sum()), "minute_coverage": cover,
          "day_ok": bool(cover >= 0.80 and out["close"].notna().all())}
    return out, st
BDIR = os.path.join(BASE, "data_bybit", "minute")
def get_day_bybit(sym, date):
    d_ = os.path.join(BDIR, sym); os.makedirs(d_, exist_ok=True)
    pq, js = os.path.join(d_, f"{date}.parquet"), os.path.join(d_, f"{date}.json")
    if os.path.exists(js):
        st = json.load(open(js)); return (pd.read_parquet(pq) if os.path.exists(pq) else None), st
    df, st = build_day_bybit(sym, date)
    if df is not None:
        df.to_parquet(pq)
    json.dump(_clean(st), open(js, "w"))
    return df, st

CTRL7 = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60"]
IMPACT = ["kappa_hat", "kappa_sq", "kyle_lam", "amihud_60"]
H_GUARD = 60
def impulse_2s(a, kappa, eta, q, L=60):
    """Price response to a unit flow shock in the depth-free model (T6 with phi = eps = 0)."""
    x = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        x[t] = (a * x[t - 1] if t else 0) + kappa * ((1.0 if t == 0 else 0.0) + f[t])
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x
def day_blocks7(s, d, flow_scale):
    df = PANEL[(s, d)]
    p = np.log(df["close"].to_numpy()) * 1e4; u = df["flow"].to_numpy() / flow_scale; vol = df["vol"].to_numpy()
    x = p - pd.Series(p).ewm(span=60, adjust=False).mean().to_numpy(); dp = np.diff(p, prepend=np.nan)
    rows = []
    for hr in range(6, 24):
        t = hr * 60; w, h, f = slice(t - 360, t), slice(t - 60, t), slice(t, min(t + 61, 1440))
        row = {"symbol": s, "date": d, "hour": hr}
        bp0 = ols(dp[w][1:], np.column_stack([u[w][1:], x[w][:-1]]))
        bm0 = ols(u[w][1:], np.column_stack([u[w][:-1], x[w][:-1]]))
        if bp0 is not None and bm0 is not None and abs(1 - bm0[0]) > 1e-6:
            k0, a0, e0 = bp0[0], 1 + bp0[1], 1 - bm0[0]; q0 = bm0[1] / e0
            imp = impulse_2s(a0, k0, e0, q0, H_GUARD); ok = np.all(np.isfinite(imp)) and abs(imp[0]) > 1e-12
            row.update({"C3_flow": k0 ** 2 * e0 * q0 - (1 - a0) * k0, "term_momentum": k0 ** 2 * e0 * q0,
                        "term_reversion": (1 - a0) * k0, "kappa_hat": k0,
                        "S60": float(np.sum(np.abs(imp))) if ok else np.nan, "P60": float(np.max(np.abs(imp))) if ok else np.nan,
                        "gain_multiple": float(np.sum(np.abs(imp)) / abs(imp[0])) if ok else np.nan})
        uu, yy = u[w][1:], dp[w][1:]; okk = np.isfinite(uu) & np.isfinite(yy)
        v60 = np.nansum(vol[h]); au = np.abs(u[w]); au = au[np.isfinite(au)]
        row.update({"rv_60": np.nanstd(np.diff(p[h])), "rv_360": np.nanstd(np.diff(p[w])), "abs_ret_60": abs(p[t - 1] - p[t - 60]),
                    "flow_imb_60": np.nansum(u[h]) / (np.nansum(np.abs(u[h])) + 1e-12), "log_vol_60": np.log1p(v60),
                    "kyle_lam": float(np.polyfit(uu[okk], yy[okk], 1)[0]) if okk.sum() > 60 and np.std(uu[okk]) > 0 else np.nan,
                    "amihud_60": np.log(abs(p[t - 1] - p[t - 60]) / v60 + 1e-9) if v60 > 0 else np.nan,
                    "U6": float(np.quantile(au, 0.95)) if len(au) > 60 else np.nan,
                    "MAE_60": float(np.nanmax(np.abs(p[f] - p[t - 1])))})
        rows.append(row)
    return rows
GUARD_RULES = {"guard_summed": lambda D: D["U6"] * D["S60"], "guard_peak": lambda D: D["U6"] * D["P60"],
               "vol": lambda D: D["rv_360"], "static": lambda D: pd.Series(1.0, index=D.index)}
def ols(y, X):
    ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 20:
        return None
    return np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), X[ok]]), y[ok], rcond=None)[0][1:]
def impulse_t6(a, phi, kappa, r, eps, eta, q, L=120):
    x = np.zeros(L); dd = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        tot = (1.0 if t == 0 else 0.0) + f[t]
        x[t] = (a * x[t - 1] + phi * dd[t - 1] if t else 0) + kappa * tot
        dd[t] = ((1 - r) * dd[t - 1] if t else 0) + eps * tot
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x
def auc_(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    rk = stats.rankdata(np.concatenate([pos, neg]))
    return (rk[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))
def cw_test(y, fb, ff, lags=6):
    adj = (y - fb) ** 2 - ((y - ff) ** 2 - (fb - ff) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    t_ = m / np.sqrt(lrv / n)
    return float(t_), float(1 - stats.norm.cdf(t_))
def design(D, cols, syms):
    X = [np.ones(len(D))] + [D[c].to_numpy() for c in cols] + [(D["symbol"] == s).to_numpy(float) for s in syms[1:]]
    return np.column_stack(X)
def winsor_train(D, cols):
    D = D.copy()
    for c in cols:
        lo, hi = D.loc[D["split"] == "train", c].quantile([0.01, 0.99])
        D[c] = D[c].clip(lo, hi)
    return D
def evaluate_g(sub, outcome, cands, ctrl, B=2000, level=0.95, seed=88):
    """Out-of-sample evaluation, as in Parts 4-5, for one or more candidates on top of `ctrl`.
    Candidates are winsorized at training 1%/99%; symbol fixed effects; date-block bootstrap."""
    cands = [cands] if isinstance(cands, str) else list(cands)
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    cw = []
    for c in cands:
        lo, hi = A0[c].quantile([0.01, 0.99]); A0[c + "_w"] = A0[c].clip(lo, hi); B0[c + "_w"] = B0[c].clip(lo, hi); cw.append(c + "_w")
    need = ctrl + cw + [outcome]
    A = A0.dropna(subset=need); Bt = B0.dropna(subset=need)
    Bt = Bt[Bt["symbol"].isin(set(A["symbol"]))].sort_values(["date", "symbol", "hour"])
    if len(A) < 200 or len(Bt) < 200 or A[outcome].std() < 1e-8 or any(A[c].std() < 1e-12 for c in cw):
        return {"skipped": "too few blocks or no variation", "n_train": len(A), "n_test": len(Bt),
                "clark_west_p": np.nan, "delta_r2": np.nan, "delta_r2_ci": [np.nan, np.nan]}
    syms = sorted(A["symbol"].unique()); y = Bt[outcome].to_numpy()
    bb = np.linalg.lstsq(design(A, ctrl, syms), A[outcome].to_numpy(), rcond=None)[0]
    bf = np.linalg.lstsq(design(A, ctrl + cw, syms), A[outcome].to_numpy(), rcond=None)[0]
    fb = design(Bt, ctrl, syms) @ bb; ff = design(Bt, ctrl + cw, syms) @ bf
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2)
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(seed); dd = Bt["date"].to_numpy(); ud = np.unique(dd)
    pos = {d: np.where(dd == d)[0] for d in ud}; dR2, dA = [], []
    for _ in range(B):
        idx = np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])
        s_ = np.sum((y[idx] - ybar) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    qa = (1 - level) / 2
    return {"n_train": len(A), "n_test": len(Bt), "n_symbols": len(syms), "n_test_dates": len(ud),
            "r2_base": 1 - np.sum((y - fb) ** 2) / sst, "r2_full": 1 - np.sum((y - ff) ** 2) / sst,
            "delta_r2": (np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst,
            "delta_r2_ci": [float(np.nanquantile(dR2, qa)), float(np.nanquantile(dR2, 1 - qa))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, qa)), float(np.nanquantile(dA, 1 - qa))],
            "clark_west_t": t_, "clark_west_p": p_, "ci_level": level,
            "coef_candidates_train": {c: float(bf[len(ctrl) + 1 + i]) for i, c in enumerate(cands)}}
def es95(v):
    v = np.asarray(v, float); q = np.quantile(v, 0.95); return float(v[v >= q].mean())
def guard_frame(D, extra_rules=None):
    rules = dict(GUARD_RULES); rules.update(extra_rules or {})
    G = D[["symbol", "date", "hour", "split", "MAE_60"]].copy()
    for r, fn in rules.items():
        G["R_" + r] = fn(D).astype(float).where(lambda v: v > 0)
    G = G.dropna(subset=["MAE_60"] + ["R_" + r for r in rules])
    for r in rules:                                          # winsorize risk at pooled training 1%/99%
        lo, hi = G.loc[G["split"] == "train", "R_" + r].quantile([0.01, 0.99])
        G["R_" + r] = G["R_" + r].clip(lo, hi)
        inv = 1.0 / G["R_" + r]
        c = inv[G["split"] == "train"].groupby(G["symbol"]).mean()
        G["q_" + r] = inv / G["symbol"].map(c)
        G["L_" + r] = G["q_" + r] * G["MAE_60"]
    G = G[G["symbol"].isin(c.index)]
    thr = G[G["split"] == "train"].groupby("symbol")["L_static"].quantile(0.95)
    for r in rules:
        G["breach_" + r] = (G["L_" + r] > G["symbol"].map(thr)).astype(float)
    return G, list(rules)
def guard_compare(G, rules, ref="vol", B=2000, level=0.95, seed=77):
    T = G[G["split"] == "test"].sort_values(["date", "symbol", "hour"])
    tlr = lambda X, r: es95(X["L_" + r]) / X["q_" + r].mean()
    point = {r: {"TLR": tlr(T, r), "mean_q": float(T["q_" + r].mean()), "breach_rate": float(T["breach_" + r].mean()),
                 "ES95_loss": es95(T["L_" + r])} for r in rules}
    Rb = rng(seed); dd = T["date"].to_numpy(); ud = np.unique(dd); pos = {d: np.where(dd == d)[0] for d in ud}
    bs = {r: [] for r in rules if r != ref}
    for _ in range(B):
        X = T.iloc[np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])]
        tr_ = tlr(X, ref)
        for r in bs:
            bs[r].append(np.log(tlr(X, r)) - np.log(tr_))
    qa = (1 - level) / 2
    comp = {r: {"log_TLR_minus_ref": float(np.log(point[r]["TLR"]) - np.log(point[ref]["TLR"])),
                "ci": [float(np.quantile(v, qa)), float(np.quantile(v, 1 - qa))],
                "share_boot_below_0": float(np.mean(np.array(v) < 0))} for r, v in bs.items()}
    return {"n_test_hours": len(T), "n_test_dates": len(ud), "ref": ref, "level": level, "point": point, "vs_ref": comp}
THETA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
def choose_theta(D, gm_col="gain_multiple"):
    """Amplification-adjusted volatility limit R = rv_360 * GM^theta, theta chosen on TRAINING data only
    (minimum training TLR). theta = 0 is the plain volatility limit."""
    res = {}
    for th in THETA_GRID:
        G, _ = guard_frame(D, {"amp_vol": (lambda X, th=th: X["rv_360"] * X[gm_col] ** th)})
        Tr = G[G["split"] == "train"]
        res[th] = es95(Tr["L_amp_vol"]) / Tr["q_amp_vol"].mean()
    return min(res, key=res.get), res


## Symbols and pre-registration (before any Bybit data)

In [ ]:
# ============================================================================
# CELL W2 — SYMBOLS: the 20 Binance thin-set coins, on Bybit (kept if Bybit has trades in 2023 and 2026)
# Fixed before any Bybit data are analysed. Thin status on Bybit is checked from training volume only.
# ============================================================================
X3 = json.load(open(os.path.join(RESULTS, "X3_T6_cross_section.json")))["metrics"]["by_symbol"]
Y3 = json.load(open(os.path.join(RESULTS, "Y3_groups.json")))["metrics"]["group"]
CAND = sorted([s for s, v in X3.items() if v["tercile"] == "thin"] + [s for s, g in Y3.items() if g == "thin"])
def bb_span(s):
    return all(_http(bb_url(s, d), "HEAD", timeout=30)[0] == 200 for d in ["2023-03-10", "2024-12-24", "2026-06-10"])
with ThreadPoolExecutor(THREADS) as ex:
    avail = dict(zip(CAND, ex.map(bb_span, CAND)))
SYMBOLS = [s for s in CAND if avail[s]]
print(f"{len(SYMBOLS)} of {len(CAND)} thin-set coins have Bybit trades over 2023-2026:", SYMBOLS)
print("missing on Bybit:", [s for s in CAND if not avail[s]])
assert len(SYMBOLS) >= 8, "too few symbols on Bybit; tell Claude"
record("W1_symbols", None, "audit", {"candidates": CAND, "available": avail, "symbols": SYMBOLS})


In [ ]:
# ============================================================================
# CELL W3 — PRE-REGISTRATION (before any Bybit data are downloaded; W4 refuses to run without it)
# ============================================================================
PREREG_P7 = {
    "question": "Second venue: does the depth-free amplification statistic forecast next-hour adverse excursions "
                "on Bybit USDT perpetuals of the Binance thin-set coins?",
    "data": "Bybit public trade archive (public.bybit.com/trading), never used before. Minute bars: last trade price, "
            "signed taker notional, notional volume. Day kept if >= 80% of minutes have trades.",
    "symbols": SYMBOLS,
    "train": "10th and 24th of each month, 2023-2024", "test": "3rd, 10th, 17th, 24th of each month, 2025-01-01 to 2026-09-30",
    "unit": "symbol-hour blocks, hours 6-23 UTC; features [t-360, t); outcome (t, t+60]",
    "statistic": "C3_flow = kappa^2 eta q - (1-a) kappa from dp = kappa u - (1-a) x and u_t = (1-eta) u_t-1 + eta q x_t-1 (T6 without depth)",
    "controls": CTRL7 + IMPACT + ["symbol fixed effects"],
    "primary": {"sample": "all symbols above", "outcome": "MAE_60", "candidate": "C3_flow (winsorized at train 1%/99%)",
                "success": "Clark-West one-sided p < 0.05 AND 95% date-block bootstrap CI of dR2 above 0"},
    "secondary": ["MAE | term_momentum", "MAE | term_reversion",
                  "position-sizing backtest, same definitions as Part 6 B: amp_vol (R = rv_360 * GM^theta, GM = S60/|h_0| from the "
                  "depth-free impulse response, theta chosen on Bybit training data) vs vol",
                  "guard_summed vs vol", "guard_peak vs vol", "static vs vol"],
    "inference": "Clark-West (Newey-West 6 lags); 2,000 date-block bootstrap draws; secondary Holm-adjusted",
    "runs": "W6 is run once; every run is counted in prereg/test_set_unlocks_p7.json",
}
spec_txt7 = json.dumps(PREREG_P7, indent=2, sort_keys=True)
P7_HASH = hashlib.sha256(spec_txt7.encode()).hexdigest()
p7_path = os.path.join(PREREG, "P7_bybit.json")
if not os.path.exists(p7_path):
    open(p7_path, "w").write(spec_txt7); print("pre-registration written:", p7_path)
elif hashlib.sha256(open(p7_path).read().encode()).hexdigest() != P7_HASH:
    raise RuntimeError("The spec differs from the registered file. Do not edit the spec after registering it.")
print("spec SHA-256:", P7_HASH)
if IN_COLAB:
    from google.colab import files
    files.download(p7_path)


## Data and blocks

In [ ]:
# ============================================================================
# CELL W4 — DOWNLOAD BYBIT TRADES AND BUILD MINUTE BARS (cached in Drive)
# ============================================================================
assert os.path.exists(p7_path) and hashlib.sha256(open(p7_path).read().encode()).hexdigest() == P7_HASH, "run W3 first"
jobs = [(s, d) for s in SYMBOLS for d in TRAIN_D + TEST_D]
print(f"{len(jobs)} symbol-days")
PANEL, VST = {}, []
with ThreadPoolExecutor(THREADS) as ex:
    futs = {ex.submit(get_day_bybit, s, d): (s, d) for s, d in jobs}
    for i, f in enumerate(as_completed(futs), 1):
        s, d = futs[f]
        try:
            df, st = f.result(); VST.append(st)
            if df is not None:
                PANEL[(s, d)] = df
        except Exception as e:
            VST.append({"symbol": s, "date": d, "reason": f"error: {e}"})
        if i % 300 == 0:
            print(f"   {i}/{len(jobs)}")
vst = pd.DataFrame(VST)
vst["day_ok"] = vst.get("day_ok", pd.Series(False, index=vst.index)).fillna(False).astype(bool)
OK = sorted([(r.symbol, r.date) for r in vst.itertuples() if r.day_ok and (r.symbol, r.date) in PANEL])
FLOW_SCALE = {}
for s in SYMBOLS:
    v = [np.nanmedian(np.abs(PANEL[(s, d)]["flow"].to_numpy())) for d in TRAIN_D if (s, d) in OK]
    v = [x for x in v if np.isfinite(x) and x > 0]
    FLOW_SCALE[s] = float(np.median(v)) if v else 1.0
pd.DataFrame(DL_LOG).to_csv(os.path.join(RESULTS, "p7_manifest_sha256.csv"), index=False)
record("W2_build", None, "audit", {"requested": len(jobs), "valid": len(OK), "flow_scale": FLOW_SCALE,
       "by_symbol": vst.groupby("symbol")["day_ok"].agg(["sum", "count"]).to_dict("index")})


In [ ]:
# ============================================================================
# CELL W5 — HOURLY BLOCKS
# ============================================================================
BLK7 = os.path.join(BASE, "data_bybit", f"blocks_p7_{len(SYMBOLS)}sym.parquet")
if os.path.exists(BLK7) and set(map(tuple, pd.read_parquet(BLK7)[["symbol", "date"]].drop_duplicates().to_numpy())) >= set(OK):
    WB = pd.read_parquet(BLK7)
else:
    rows = []
    for i, (s, d) in enumerate(OK):
        rows += day_blocks7(s, d, FLOW_SCALE[s])
        if (i + 1) % 300 == 0:
            print(f"   blocks: {i+1}/{len(OK)}")
    WB = pd.DataFrame(rows); WB.to_parquet(BLK7)
WB["split"] = np.where(WB["date"] <= TRAIN_END, "train", "test")
WB["kappa_sq"] = WB["kappa_hat"] ** 2
WB = winsor_train(WB, IMPACT)
print(WB.groupby("split").size())
record("W3_blocks", None, "audit", {"blocks": WB.groupby("split").size().to_dict(),
       "coverage": {c: float(WB[c].notna().mean()) for c in ["C3_flow", "S60", "U6"] + IMPACT}})


## The test: run ONCE (it also zips and downloads the results)

In [ ]:
# ============================================================================
# CELL W6 — THE TEST (RUN ONCE; every run is counted)
# ============================================================================
assert hashlib.sha256(open(p7_path).read().encode()).hexdigest() == P7_HASH, "spec changed after registration"
pth7 = os.path.join(PREREG, "test_set_unlocks_p7.json")
n_unl7 = (json.load(open(pth7))["n"] if os.path.exists(pth7) else 0) + 1
json.dump({"n": n_unl7, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": P7_HASH}, open(pth7, "w"))
def holm(pd_):
    items = sorted((v if np.isfinite(v) else 1.0, k) for k, v in pd_.items()); m = len(items); out, run = {}, 0.0
    for j, (p_, k) in enumerate(items):
        run = max(run, min(1.0, p_ * (m - j))); out[k] = run
    return out
C7 = CTRL7 + IMPACT
prim7 = evaluate_g(WB, "MAE_60", "C3_flow", C7)
ok7 = bool(np.isfinite(prim7.get("clark_west_p", np.nan)) and prim7["clark_west_p"] < 0.05 and prim7["delta_r2_ci"][0] > 0)
sec7 = {"MAE | term_momentum": evaluate_g(WB, "MAE_60", "term_momentum", C7),
        "MAE | term_reversion": evaluate_g(WB, "MAE_60", "term_reversion", C7)}
THETA7, theta7_tr = choose_theta(WB)
print("theta chosen on training data:", THETA7)
G7, rules7 = guard_frame(WB, {"amp_vol": (lambda X: X["rv_360"] * X["gain_multiple"] ** THETA7)})
gb7 = guard_compare(G7, rules7, ref="vol")
p7s = {k: v.get("clark_west_p", np.nan) for k, v in sec7.items()}
for r in ["amp_vol", "guard_summed", "guard_peak", "static"]:
    p7s[f"{r} vs vol"] = 1 - gb7["vs_ref"][r]["share_boot_below_0"]
h7 = holm(p7s)
for k in sec7:
    sec7[k]["holm_p"] = h7[k]
sh = lambda v: {k: (round(v[k], 5) if isinstance(v.get(k), float) else v.get(k)) for k in
                ["n_train", "n_test", "n_symbols", "delta_r2", "delta_r2_ci", "delta_auc_ci", "clark_west_t", "clark_west_p", "holm_p", "coef_candidates_train", "skipped"] if k in v}
print("=== PRIMARY (Bybit, C3_flow):", "PASS" if ok7 else "FAIL"); print(sh(prim7))
for k, v in sec7.items():
    print(k, sh(v))
print(pd.DataFrame(gb7["point"]).T.round(4).to_string())
for r, v in gb7["vs_ref"].items():
    print(f"   {r:13s} log TLR - log TLR_vol = {v['log_TLR_minus_ref']:+.4f}  CI {np.round(v['ci'], 4)}  Holm p {h7[f'{r} vs vol']:.4f}")
record("W6_bybit_test", ok7, "real-data", {"primary": prim7, "secondary": sec7, "guard": gb7, "theta": THETA7, "theta_train_TLR": theta7_tr, "holm_p": h7,
       "prereg_sha256": P7_HASH, "test_set_unlocks": n_unl7}, criterion=PREREG_P7["primary"]["success"])
zp7 = os.path.join(BASE, "p7_results.zip")
with zipfile.ZipFile(zp7, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "W*.json")) + [os.path.join(RESULTS, "p7_manifest_sha256.csv"), p7_path, pth7]:
        if os.path.exists(fp):
            z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zp7)
if IN_COLAB:
    from google.colab import files
    files.download(zp7)


If W0 says the archive is not reachable, Bybit is blocking this Colab region. Tell me and I'll adapt it.

Send me `p7_results.zip`.